In [1]:
# ================= Cell 1 =================
# 安裝必要的套件 (如果在 Colab 執行，建議先執行這行；Kaggle 通常已內建)
# !pip install transformers datasets scikit-learn pandas torch

import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW 
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm

# 設定隨機種子以確保結果可重現
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [2]:
# ================= Cell 2 (修改後) =================
import os
import pandas as pd
from sklearn.model_selection import StratifiedKFold # 🎯 引入 K-Fold
from sklearn.preprocessing import LabelEncoder

DATA_DIR = '/kaggle/input/competitions/map-charting-student-math-misunderstandings/' 

train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

# 確保空值變成字串 'NA'
train_df['Misconception'] = train_df['Misconception'].fillna('NA')

# 將 Category 和 Misconception 合併
train_df['target_label'] = train_df['Category'] + ':' + train_df['Misconception']

# 將 Question, Answer, Explanation 拼接成一個完整的上下文
def create_input_text(row):
    return f"Question: {row['QuestionText']} [SEP] Answer: {row['MC_Answer']} [SEP] Explanation: {row['StudentExplanation']}"

train_df['input_text'] = train_df.apply(create_input_text, axis=1)
test_df['input_text'] = test_df.apply(create_input_text, axis=1)

# Label Encoding
label_encoder = LabelEncoder()
train_df['label'] = label_encoder.fit_transform(train_df['target_label'])
num_labels = len(label_encoder.classes_)

print(f"Total number of full classes (Category:Misconception): {num_labels}")

# 🎯 移除原本的 train_test_split，改為初始化 5-Fold 分層抽樣
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Total number of full classes (Category:Misconception): 65


In [3]:
# ================= Cell 3 & 4 (合併為 K-Fold 訓練核心) =================
from transformers import get_cosine_schedule_with_warmup # 🎯 改用 Cosine
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler # 🎯 引入混合精度加速

MODEL_NAME = '/kaggle/input/datasets/huangtzuchen/my-mathbert-weights/mathbert-local'
MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 4

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class MathMisconceptionDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=256):
        self.texts = texts.values
        self.labels = labels.values if labels is not None else None
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        inputs = self.tokenizer(
            text, add_special_tokens=True, max_length=self.max_len,
            padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt'
        )
        item = {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten()
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# 🎯 啟用標籤平滑 (Label Smoothing)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)
scaler = GradScaler() # 初始化 FP16 梯度縮放器

def train_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    correct_preds = 0

    for batch in tqdm(dataloader, desc="Training", leave=False):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        # 🎯 使用 autocast 進行 FP16 混合精度前向傳播
        with autocast():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            loss = loss_fn(logits, labels) # 使用自訂的平滑 Loss

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct_preds += torch.sum(preds == labels).item()

        # 🎯 使用 scaler 進行反向傳播與優化
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

    return total_loss / len(dataloader), correct_preds / len(dataloader.dataset)

def eval_model(model, dataloader, device):
    model.eval()
    total_loss = 0
    correct_preds = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            loss = loss_fn(logits, labels)

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            correct_preds += torch.sum(preds == labels).item()

    return total_loss / len(dataloader), correct_preds / len(dataloader.dataset)

# ================= 🚀 開始 5-Fold 迴圈 =================
for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['label'])):
    print(f"\n{'='*20} 🚀 開始訓練 Fold {fold + 1}/5 {'='*20}")
    
    # 1. 根據 K-Fold 索引切分資料
    train_data = train_df.iloc[train_idx]
    val_data = train_df.iloc[val_idx]
    
    # 2. 建立 DataLoader
    train_dataset = MathMisconceptionDataset(train_data['input_text'], train_data['label'], tokenizer, MAX_LEN)
    val_dataset = MathMisconceptionDataset(val_data['input_text'], val_data['label'], tokenizer, MAX_LEN)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # 3. 初始化全新模型 (確保每個 Fold 都是從頭訓練)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=num_labels, ignore_mismatched_sizes=True
    ).to(device)
    
    # 4. 設定優化器與 Cosine Scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01) # LR 稍微調高給 Cosine 空間
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps*0.1), num_training_steps=total_steps
    )
    
    # 5. 開始執行 Epoch
    best_acc = 0
    for epoch in range(EPOCHS):
        print(f"Epoch {epoch+1}/{EPOCHS}")
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")

        val_loss, val_acc = eval_model(model, val_loader, device)
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            # 🎯 為每個 Fold 儲存獨立的權重檔案
            torch.save(model.state_dict(), f'best_mathbert_fold{fold+1}.pt')
            print(f">> Saved Best Model for Fold {fold+1}!")


==================== 🚀 開始訓練 Fold 1/5 ====================


/tmp/ipykernel_23/4238193629.py:38: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() # 初始化 FP16 梯度縮放器
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/huangtzuchen/my-mathbert-weights/mathbert-local
Key               | Status   |                                                                                      
------------------+----------+--------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([2]) vs model:torch.Size([65])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([2, 768]) vs model:torch.Size([65, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Epoch 1/4


...

Val Loss: 0.7680 | Val Acc: 0.8704
>> Saved Best Model for Fold 5!
Epoch 4/4


Training:   0%|          | 0/1835 [00:00<?, ?it/s]

Train Loss: 0.5771 | Train Acc: 0.9400


Evaluating:   0%|          | 0/459 [00:00<?, ?it/s]

Val Loss: 0.8068 | Val Acc: 0.8723
>> Saved Best Model for Fold 5!


In [4]:
# ================= Cell 5 (推論與 Ensemble 修改後) =================
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from torch.cuda.amp import autocast

# 準備 Test Loader
test_dataset = MathMisconceptionDataset(test_df['input_text'], tokenizer=tokenizer, max_len=MAX_LEN)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

all_top_3_preds = []

# 🎯 建立一個列表，用來儲存 5 個 Fold 模型預測的機率矩陣
fold_probs = []

# 依序讀取 5 個 Fold 的模型權重進行推論
for fold in range(1, 6):
    print(f"正在執行 Fold {fold} 模型推論...")
    model.load_state_dict(torch.load(f'best_mathbert_fold{fold}.pt'))
    model.eval()
    
    current_fold_probs = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"Fold {fold} Inference", leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            with autocast():
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                probs = torch.softmax(outputs.logits, dim=-1)
            
            current_fold_probs.append(probs.cpu())
            
    # 將這個 Fold 所有的預測 batch 拼起來，存入列表中
    fold_probs.append(torch.cat(current_fold_probs, dim=0))

# 🎯 將 5 個模型的預測機率相加並取平均 (關鍵 Ensemble 步驟！)
avg_probs = torch.mean(torch.stack(fold_probs), dim=0)

# 取平均後機率最高的 Top 3
top_3_indices = torch.topk(avg_probs, 3, dim=1).indices.numpy()

for indices in top_3_indices:
    top_3_labels = label_encoder.inverse_transform(indices)
    all_top_3_preds.append(" ".join(top_3_labels))

# 建立官方格式的 submission.csv
submission_df = pd.DataFrame({
    'row_id': test_df['row_id'] if 'row_id' in test_df.columns else test_df['QuestionId'].astype(str) + "_" + test_df['MC_Answer'].astype(str),
    'Category:Misconception': all_top_3_preds
})

submission_df.to_csv('submission.csv', index=False)
print("🎉 Submission 檔案已包含 5-Fold Ensemble 結果，成功儲存為 submission.csv")

正在執行 Fold 1 模型推論...


Fold 1 Inference:   0%|          | 0/1 [00:00<?, ?it/s]

/tmp/ipykernel_23/3971860572.py:28: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


正在執行 Fold 2 模型推論...


Fold 2 Inference:   0%|          | 0/1 [00:00<?, ?it/s]

...

正在執行 Fold 4 模型推論...


Fold 4 Inference:   0%|          | 0/1 [00:00<?, ?it/s]

正在執行 Fold 5 模型推論...


Fold 5 Inference:   0%|          | 0/1 [00:00<?, ?it/s]

🎉 Submission 檔案已包含 5-Fold Ensemble 結果，成功儲存為 submission.csv
